In [91]:
import json
import datetime
import time

In [92]:
def two_hours_forward(timestamp_string: str):
    date, time = timestamp_string.split(' ')
    year, month, day = date.split('-')
    hour, minute, second = time.split(':')
    date = datetime.datetime(int(year), int(month), int(day), int(hour), int(minute), int(second))
    date += datetime.timedelta(hours=2)
    return date

In [93]:
def get_weekday(timestamp_string: str):
    date, time = timestamp_string.split(' ')
    year, month, day = date.split('-')
    hour, minute, second = time.split('.')
    weekday = datetime.datetime(int(year), int(month), int(day), int(hour), int(minute), int(second)).weekday()
    return weekday

In [94]:
def time_to_integer(timestamp_string: str):
    output = 0
    date, time = timestamp_string.split(' ')
    hour, minute, _ = time.split('.')
    hour, minute = int(hour), int(minute)
    output += 60 * hour + minute
    return output

In [95]:
# Dictionary representing all stations and freestanding bikes for every 5 minutes
timestamp_dictionary = dict()

# Extracting stations info for stations in Warsaw
with open('json_files/2024-08-18to25-veturilo.json', 'r', encoding='utf8') as file:
    for line in file:
        data_piece = json.loads(line)
        if data_piece['city_uid'] == 812:
            timestamp = str(two_hours_forward(data_piece['timestamp']['$date'].replace('T', ' ')[:-1])).replace(':', '.')
            if not timestamp in timestamp_dictionary.keys():
                timestamp_dictionary[timestamp] = []
            timestamp_dictionary[timestamp].append(data_piece)

In [96]:
with open('weather_data.json') as json_file:
    weather_details = json.load(json_file)

In [97]:
attributes_other = []
attributes_friday = []
attributes_saturday = []
attributes_sunday = []

a =  0
for timestamp in timestamp_dictionary.keys():
    for bike_stand in timestamp_dictionary[timestamp]:
        if 20.9 < bike_stand['lng'] < 21.2 and 52.05 < bike_stand['lat'] < 52.35 and bike_stand['bike']:
            single_attribute = [bike_stand['lng'],
                                bike_stand['lat'],
                                # time_to_integer(timestamp),
                                # weather_details[timestamp[:10]]['avg_temperature'],
                                # weather_details[timestamp[:10]]['rain']
                                ]
            for _ in range(bike_stand['bikes']):
                if get_weekday(timestamp) < 4:
                    attributes_other.append(single_attribute)
                elif get_weekday(timestamp) == 4:
                    attributes_friday.append(single_attribute)
                elif get_weekday(timestamp) == 5:
                    attributes_saturday.append(single_attribute)
                else:
                    attributes_sunday.append(single_attribute)

filenames = ["other.txt", "friday.txt", "saturday.txt", "sunday.txt"]
all_attributes = [attributes_other, attributes_friday, attributes_saturday, attributes_sunday]

for i in range(len(all_attributes)):
    with open(f'data/{filenames[i]}', 'w') as file:
        for attribute_line in all_attributes[i]:
            for attribute in attribute_line:
                file.write(str(attribute) + ', ')
            file.write('\n')

In [109]:
filenames = ["other.txt", "friday.txt", "saturday.txt", "sunday.txt"]
all_attributes = [attributes_other, attributes_friday, attributes_saturday, attributes_sunday]

file_all_data = open(f'data/all_data.txt', 'w')

for i in range(len(all_attributes)):
    file_detailed = open(f'data/{filenames[i]}', 'w')
    for attribute_line in all_attributes[i]:
        for attribute in attribute_line:
            file_detailed.write(str(attribute) + ', ')
            file_all_data.write(str(attribute) + ', ')
        file_detailed.write('\n')
        file_all_data.write('\n')
    file_detailed.close()
file_all_data.close()


In [110]:
print(time_to_integer('2024-08-25 20.25.00'))

1225


In [83]:
print(len(attributes))

6022434
